In [1]:
import numpy as np
import matplotlib.pylab as plt
%matplotlib inline
import healpy as hp
import rubin_sim.splat as splat
import copy 

import sqlite3
import pandas as pd
from os.path import basename

import glob
from becker_desi_stripes import does_it_work_for_desi, get_desi_stripes

Could not make DESI footprint functions!


In [2]:
class CountsTimeElement(splat.BaseReportElement):
    def compute_elements(self,visits_df=None, visits_array=None, nside=32, band=None):
        metrics = []
        infos = []
        times = np.arange(3653)
        metrics.append(splat.AccumulateCountMetric(times))
        info = {"data_source": self.run_name}
        if band is None:
            info["observations_subset"] = "\nNo ToO, No DDf, expt > 20"
        else:
            info["observations_subset"] =  band + "\nNo ToO, No DDf, expt > 20"

        # Number of visits
        temp_info = copy.copy(info)
        temp_info["group"] = "Basics"
        temp_info["subgroup"] = "Counts"
        infos.append(temp_info)
       
        is_too = np.array(["ToO" in note for note in visits_array["scheduler_note"]])
        is_ddf = np.array(["DD" in note for note in visits_array["scheduler_note"]])

        if band is None:
            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20))
        else:
            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20) & (visits_array["band"] == band))
        sub_data = visits_array[indx]
        sl = splat.Slicer(nside=nside, missing=0)
        hp_arrays, infos = sl(sub_data, metrics, info=infos)

        return hp_arrays, infos, None

In [3]:
db_files = ['baseline_v5.3.7_10yrs.db', 'three_stripes_v5.3.7_10yrs.db',
            'three_stripes_kr_v5.3.7_10yrs.db',
            'desi_tskr_v5.3.7_10yrs.db',
            'desi_stripes_delayed_v5.3.7_10yrs.db',
            'desi_stripes_v5.3.7_10yrs.db',
            "double_stripes_timing_bands_v5.3.7_10yrs.db",
            "three_stripes_or_v5.3.7_10yrs.db",
            "delay_roll_v5.3.7_10yrs.db"
            ]

In [4]:

works_bands_stripes = {}
works_bands_stripes_complete = {}
works_bands_stripes_swapped = {}

for filename in db_files:
    counts_over_time = {}
    element = CountsTimeElement(observations=filename)
    for band in 'ugrizy':   
        hp_arrays, infos, summary = element(band="u")
        counts_over_time[band] = hp_arrays

    count_maps = {}
    for band in 'ugrizy':
        for i in np.arange(10)+1:
            count_maps["%i %s" % (i, band)] = counts_over_time[band][:,int(365.25*i)]

    what_works, values = does_it_work_for_desi(count_maps, get_desi_stripes(), tol=5)
    works_bands_stripes[filename.replace(".db", "")] = np.sum(list(what_works.values()))

    what_works, values = does_it_work_for_desi(count_maps, get_desi_stripes(), tol=5, use_per_stripe_u_threshold=True)
    works_bands_stripes_complete[filename.replace(".db", "")] = np.sum(list(what_works.values()))

    what_works, values = does_it_work_for_desi(count_maps, get_desi_stripes(), tol=5,
                                               use_per_stripe_u_threshold=True, )
    works_bands_stripes_complete[filename.replace(".db", "")] = np.sum(list(what_works.values()))

    

In [5]:
works_bands_stripes

{'baseline_v5.3.7_10yrs': np.int64(10),
 'three_stripes_v5.3.7_10yrs': np.int64(14),
 'three_stripes_kr_v5.3.7_10yrs': np.int64(16),
 'desi_tskr_v5.3.7_10yrs': np.int64(45),
 'desi_stripes_delayed_v5.3.7_10yrs': np.int64(43),
 'desi_stripes_v5.3.7_10yrs': np.int64(57),
 'double_stripes_timing_bands_v5.3.7_10yrs': np.int64(64),
 'three_stripes_or_v5.3.7_10yrs': np.int64(13),
 'delay_roll_v5.3.7_10yrs': np.int64(14)}

In [6]:
np.size(list(what_works.values()))

85

In [7]:
works_bands_stripes_complete

{'baseline_v5.3.7_10yrs': np.int64(12),
 'three_stripes_v5.3.7_10yrs': np.int64(16),
 'three_stripes_kr_v5.3.7_10yrs': np.int64(18),
 'desi_tskr_v5.3.7_10yrs': np.int64(49),
 'desi_stripes_delayed_v5.3.7_10yrs': np.int64(47),
 'desi_stripes_v5.3.7_10yrs': np.int64(58),
 'double_stripes_timing_bands_v5.3.7_10yrs': np.int64(64),
 'three_stripes_or_v5.3.7_10yrs': np.int64(15),
 'delay_roll_v5.3.7_10yrs': np.int64(15)}